In [47]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder

In [50]:
df = pd.read_csv("final_emotion_dataset.csv")
print(df.head())
# -----------------------------
# Keep required columns
# -----------------------------

X = df["text"]
y = df["main_emotion"]

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

                                                text sentiment  \
0  I experienced this emotion when my grandfather...   sadness   
1   when I first moved in , I walked everywhere ....   neutral   
2  ` Oh ! " she bleated , her voice high and rath...     anger   
3  However , does the right hon. Gentleman recogn...      fear   
4  My boyfriend didn't turn up after promising th...   sadness   

                                          clean_text  label main_emotion  \
0        experienced emotion grandfather passed away     26          Sad   
1  first moved walked everywhere within week purs...     20      Neutral   
2             oh bleated voice high rather indignant      2        Angry   
3  however right hon gentleman recognise profound...     14         Fear   
4                boyfriend not turn promising coming     26          Sad   

   main_label  
0           8  
1           6  
2           1  
3           4  
4           8  


In [3]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(X)
# -----------------------------
# Train Test Split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded

)


In [6]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [7]:
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy\n")
print(accuracy)


Accuracy

0.39561015806427446


In [8]:
print("\nClassification Report\n")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=encoder.classes_
    )
)


Classification Report

               precision    recall  f1-score   support

    Affection       0.05      0.04      0.05      3138
        Angry       0.29      0.26      0.28      5289
    Curiosity       0.07      0.05      0.06      2019
Embarrassment       0.00      0.00      0.00       283
         Fear       0.31      0.17      0.22      3086
        Happy       0.44      0.46      0.45      7819
      Neutral       0.36      0.44      0.40     10667
       Relief       0.00      0.00      0.00       141
          Sad       0.67      0.71      0.69      7605

     accuracy                           0.40     40047
    macro avg       0.24      0.24      0.24     40047
 weighted avg       0.38      0.40      0.38     40047



In [9]:
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix\n")
print(cm)


Confusion Matrix

[[ 128  376  124   16   78  938 1309   10  159]
 [ 338 1384  197   59  223  637 1993    7  451]
 [ 129  227  104    5  154  467  848    4   81]
 [  14   65    8    0   17   43   91    0   45]
 [ 105  251  157   13  528  227  900    3  902]
 [ 663  579  355   17  146 3622 2151   35  251]
 [ 855 1374  547   44  425 1900 4710   32  780]
 [  23    9    4    0    4   57   38    0    6]
 [ 133  490   72   26  129  322 1058    8 5367]]


In [18]:
joblib.dump(
    rf,
    "../models/main_emotion_rf.pkl"
)

['../models/main_emotion_rf.pkl']

In [51]:
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
)

In [20]:
label_encoder = joblib.load("../models/sub_emotion_encoder.pkl")
df["label"] = label_encoder.transform(df["sentiment"])

print(label_encoder.classes_)

['admiration' 'amusement' 'anger' 'annoyance' 'approval' 'caring'
 'confusion' 'curiosity' 'desire' 'disappointment' 'disapproval' 'disgust'
 'embarrassment' 'excitement' 'fear' 'gratitude' 'grief' 'joy' 'love'
 'nervousness' 'neutral' 'optimism' 'pride' 'realization' 'relief'
 'remorse' 'sadness' 'surprise']


In [21]:
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print(len(val_df))

40047


In [22]:
MODEL_PATH = "../models/distilbert_sub_emotion"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [23]:
def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

val_ds = Dataset.from_pandas(
    val_df[["clean_text", "label"]]
)

val_ds = val_ds.map(
    tokenize,
    batched=True
)

val_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

Map:   0%|          | 0/40047 [00:00<?, ? examples/s]

In [24]:
trainer = Trainer(model=model)

In [25]:
pred = trainer.predict(val_ds)

c:\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [14]:
y_pred = np.argmax(pred.predictions, axis=1)
y_true = pred.label_ids

print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
) # This is distilled BERT model for sub-emotion classification. The model is trained on the training dataset and evaluated on the validation dataset. The classification report shows the precision, recall, and F1-score for each class in the sub-emotion classification task.

NameError: name 'pred' is not defined

In [13]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_true, y_pred)

print(f"Overall Accuracy: {accuracy:.4f}")
print(f"Overall Accuracy: {accuracy*100:.2f}%")

Overall Accuracy: 0.4534
Overall Accuracy: 45.34%


In [3]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

NameError: name 'y_true' is not defined

In [42]:
main_encoder = joblib.load("../models/main_emotion_encoder.pkl")

In [52]:
X = df["clean_text"]
y = df["main_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [53]:
PROJECT_ROOT = os.path.abspath("..")

main_model_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "distilbert_emotion"
)

In [54]:
main_tokenizer = AutoTokenizer.from_pretrained(main_model_path)

main_model = AutoModelForSequenceClassification.from_pretrained(
    main_model_path
)

main_model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
main_model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [58]:
from torch.utils.data import DataLoader
batch_size = 64   # You can use 32 if RAM is limited

y_pred = []

loader = DataLoader(X_test.tolist(), batch_size=batch_size)

for batch in loader:

    inputs = main_tokenizer(
        list(batch),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = main_model(**inputs)

    preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

    y_pred.extend(preds)

In [59]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=main_encoder.classes_
    )) #classifacation report for main emotion classification using distilled BERT model. The classification report shows the precision, recall, and F1-score for each class in the main emotion classification task.

               precision    recall  f1-score   support

    Affection       0.26      0.10      0.15      3138
        Angry       0.42      0.47      0.44      5289
    Curiosity       0.30      0.16      0.21      2019
Embarrassment       0.31      0.08      0.12       283
         Fear       0.67      0.47      0.55      3086
        Happy       0.56      0.65      0.61      7819
      Neutral       0.47      0.57      0.51     10667
       Relief       0.00      0.00      0.00       141
          Sad       0.77      0.77      0.77      7605

     accuracy                           0.54     40047
    macro avg       0.42      0.36      0.37     40047
 weighted avg       0.53      0.54      0.53     40047



c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
